# Finite calibration and holdout validation with Kessetsu

This notebook runs a bounded parameter study, imports two explicitly synthetic datasets, selects a candidate using only the calibration condition, and then reports the separate holdout score. Electrical simulation, interpolation, masking and scoring stay in Kessetsu Core.

> This tutorial demonstrates a reproducible software workflow. It is not a physical measurement, proof of a device mechanism, continuous optimization or evidence outside the declared grid and conditions.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from kessetsu import KessetsuClient

repo = Path.cwd()
research = repo / 'examples' / 'research'
if not (research / 'divider-fit.kessstudy.json').exists():
    raise RuntimeError('Run this notebook from the Kessetsu repository or release-bundle root')
artifacts = repo / '.artifacts' / 'fitting-notebook'
artifacts.mkdir(parents=True, exist_ok=True)
client = KessetsuClient()
print(f'Kessetsu CLI {client.version()}')

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

study_output = artifacts / 'study-results.json'
if study_output.exists():
    study_output.unlink()
study = client.run_study(
    research / 'divider-fit.kessstudy.json',
    study_output,
)
calibration = client.import_csv(
    research / 'divider-calibration.csv',
    read_json(research / 'divider-calibration.kessimport.json'),
    output=artifacts / 'calibration.kessdata.json',
    force=True,
)
validation = client.import_csv(
    research / 'divider-validation.csv',
    read_json(research / 'divider-validation.kessimport.json'),
    output=artifacts / 'validation.kessdata.json',
    force=True,
)
fit = client.evaluate_fit(
    study,
    read_json(research / 'divider-fit.kessfit.json'),
    {'calibration': calibration, 'validation': validation},
    output=artifacts / 'fit-result.json',
    force=True,
)

In [ ]:
candidates = fit.candidates_table().to_pandas()
observations = fit.observations_table().to_pandas()
selected = candidates.loc[candidates['selected']].iloc[0]
print('Selected resistance:', selected['parameter.resistance'])
print('Calibration score:', selected['calibration_score'])
print('Holdout score:', selected['validation_score'])
candidates

In [ ]:
selected_id = selected['candidate_id']
calibration_residuals = fit.comparison(selected_id, 'one-volt calibration').table('output').to_pandas()
validation_residuals = fit.comparison(selected_id, 'two-volt holdout').table('output').to_pandas()

fig, (score_axis, residual_axis) = plt.subplots(2, 1, figsize=(8, 7))
score_axis.plot(candidates['parameter.resistance'], candidates['calibration_score'], 'o-', label='Calibration')
score_axis.plot(candidates['parameter.resistance'], candidates['validation_score'], 's--', label='Holdout')
score_axis.set_ylabel('Normalized RMS')
score_axis.set_title('Every evaluated candidate is retained')
score_axis.legend()
score_axis.grid(alpha=0.25)
residual_axis.axhline(0, color='0.5', linewidth=1)
residual_axis.plot(calibration_residuals['axis'] * 1e3, calibration_residuals['residual'], 'o-', label='Calibration residual')
residual_axis.plot(validation_residuals['axis'] * 1e3, validation_residuals['residual'], 's--', label='Holdout residual')
residual_axis.set_xlabel('Time (ms)')
residual_axis.set_ylabel('Residual (V)')
residual_axis.legend()
residual_axis.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(artifacts / 'fit-summary.png', dpi=160)
fig

The artifact directory contains the complete study, normalized calibration and validation datasets, every candidate comparison, failures, point-by-point residuals, model/solver identities and the plot. Selection uses only observations marked `calibration`; `validation` is reported afterward. A boundary hit or several near-equivalent candidates is surfaced as a warning rather than hidden.